# Zero-Budget Deep Learning Training Pipeline: Google Colab
**Project:** Development and Validation of an rPPG-Based System for Non-Contact Vital Sign Monitoring  
**Target Architecture:** PhysNet 3D-CNN & MTTS-CAN on Free NVIDIA Tesla T4 GPU  
**Storage:** Google Drive Zero-Cost Synchronization  
**Financial Cost:** $0.00 (Strictly Free Allowance)

This notebook executes deep-learning model training remotely on Google Colab's free GPU compute tier, logging losses, creating PyTorch checkpoints every epoch, and synchronizing artifacts to Google Drive to ensure fault-tolerant recovery after runtime disconnects.

## 1. Hardware Verification & Environment Setup

In [ ]:
# Verify GPU allocation (Expected: Tesla T4 or P100)
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Mount Google Drive for Persistent Checkpoint & Registry Storage
Google Colab local storage is ephemeral. We mount Google Drive (free 15 GB tier) so all checkpoints (`.pt`) and model artifacts (`.onnx`) persist across session disconnects.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create project directory on Google Drive
GDRIVE_DIR = '/content/drive/MyDrive/rPPG_Research'
CHECKPOINT_DIR = os.path.join(GDRIVE_DIR, 'checkpoints')
EXPERIMENTS_DIR = os.path.join(GDRIVE_DIR, 'experiments')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)
print(f"Persistent checkpoint directory ready: {CHECKPOINT_DIR}")

## 3. Clone Repository & Install Dependencies

In [ ]:
# Clone or pull latest codebase
%cd /content
!git clone https://github.com/your-username/rppg-vital-sign-monitor.git || (cd rppg-vital-sign-monitor && git pull)
%cd /content/rppg-vital-sign-monitor/backend

# Install lightweight requirements
!pip install -q onnx scipy mediapipe

## 4. Run Training with Automatic Disconnect Recovery
The training harness automatically writes `{exp_id}_latest.pt` every epoch. If Colab disconnects, provide `resume_checkpoint` to resume from the exact epoch without losing progress.

In [ ]:
import sys
sys.path.insert(0, '/content/rppg-vital-sign-monitor/backend')

from app.models_dl.trainer import PhysNetTrainer

# Initialize GPU trainer directing checkpoints directly into Google Drive
trainer = PhysNetTrainer(
    device="cuda" if torch.cuda.is_available() else "cpu",
    checkpoint_dir=CHECKPOINT_DIR,
)

# Set hyperparameter configuration
NUM_EPOCHS = 10
NUM_SUBJECTS = 20
BATCH_SIZE = 4
EXPERIMENT_NAME = "Colab_Free_T4_PhysNet_Run"

# Check for existing latest checkpoint in Google Drive for seamless resumption
latest_ckpts = [os.path.join(CHECKPOINT_DIR, f) for f in os.listdir(CHECKPOINT_DIR) if f.endswith('_latest.pt')]
resume_path = latest_ckpts[-1] if latest_ckpts else None
if resume_path:
    print(f"Found checkpoint to resume: {resume_path}")
else:
    print("Starting new training run from scratch (Epoch 1)...")

# Execute training run with continuous Google Drive mirroring
summary = trainer.run_training_experiment(
    num_epochs=NUM_EPOCHS,
    num_subjects=NUM_SUBJECTS,
    batch_size=BATCH_SIZE,
    experiment_name=EXPERIMENT_NAME,
    resume_checkpoint=resume_path,
    backup_dir=CHECKPOINT_DIR,
)

print("Training Complete!")
print(f"Final Validation Loss: {summary['final_val_loss']}")
print(f"Test Loss: {summary['test_loss']}")
print(f"Best Checkpoint: {summary['best_checkpoint_path']}")
print(f"Exported ONNX Model: {summary['onnx_export_path']}")

## 5. Verify Checkpoints & Synchronized Artifacts in Google Drive

In [ ]:
print("Synchronized Google Drive Artifacts:")
!ls -lh "/content/drive/MyDrive/rPPG_Research/checkpoints"